# Simulation ALS Data: Two-Epoch TAM3C2 + Multi-Scale Analysis

Time-Adaptive M3C2 using `py4dgeo.tam3c2` on one reference epoch and one explicitly selected target epoch.

**Dataset:** Sand Dune TLS Scans
- **Location:** `C:\rsa\research_proj\blender_project\simulation_test2\output\simulation_test2_als`
- **Format:** XYZ files

**Workflow:**
1. Load all available epochs
2. Select one reference epoch and one target epoch
3. Sample corepoints from the reference epoch
4. Build a `TAM3C2` algorithm object with only the reference/target pair in `epochs_timeseries`
5. Run per-target TAM3C2 for the selected target epoch
6. Compare multi-scale spherical normal estimation and cylindrical distance-estimation behavior

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from datetime import datetime, timedelta

import numpy as np
import matplotlib.pyplot as plt

import py4dgeo
from py4dgeo import (
    TAM3C2,
    Weighting,
    extract_reference_and_others,
    sample_corepoints,
)
from py4dgeo.segmentation import RegionGrowingSeed, temporal_averaging
from py4dgeo.data_loader import read_pc_epochs_and_assign_timestamps

## 1. Configuration

In [ ]:
data_path = r'C:\rsa\research_proj\blender_project\simulation_test2\output\simulation_test2_als_downsampled'
output_path = os.path.join(os.getcwd(), 'simulation2_ref_target_tam3c2.zip')

# Two-epoch TAM3C2 setup: one reference epoch and one explicit target epoch.
reference_timestamp = datetime(2020, 1, 6, 0, 0, 0)
target_timestamp = datetime(2020, 1, 22, 0, 0, 0)

# TAM3C2 parameters - multi-scale grid for reference/target-only aggregation
normal_radii = [0.1, 0.2, 0.3, 0.5]
max_window_ratio = [0.1, 0.2, 0.3, 0.5]
required_points = 10
cyl_radius = 0.5
max_distance = 10.0
registration_error = 0.01
sigma_ratio = 1.0
space_time_ratio = 1.0
weighting = Weighting.GAUSSIAN
include_center_epoch = True
keep_neighborhoods = True

# 4D-OBC parameters
# obc_neighborhood_radius = 1.0
# obc_min_segments = 1
# obc_minperiod = 1
# obc_height_threshold = 0.05
# obc_thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
# obc_smoothing_window = 1

### corepoints from the reference file

In [ ]:
reference_file_path = os.path.join(os.getcwd(), 'simulation2_reference_v2.zip')
ref_analysis = py4dgeo.SpatiotemporalAnalysis(reference_file_path, force=False)
corepoints = ref_analysis.corepoints.cloud

In [ ]:
corepoints

## 2. Load epochs and select the reference/target pair

In [ ]:
epochs = read_pc_epochs_and_assign_timestamps(folder=data_path, start_time=datetime(2020, 1, 1), time_increment=timedelta(days=1))

In [ ]:
# Keep only the two epochs used by TAM3C2 aggregation.
epochs_all = sorted(epochs, key=lambda e: e.timestamp)
reference_epoch = next((e for e in epochs_all if e.timestamp == reference_timestamp), None)
target_epoch = next((e for e in epochs_all if e.timestamp == target_timestamp), None)

if reference_epoch is None:
    raise ValueError(f"Reference {reference_timestamp} not in data")

if target_epoch is None:
    raise ValueError(f"Target {target_timestamp} not in data")

if target_epoch.timestamp == reference_epoch.timestamp:
    raise ValueError("target_timestamp must differ from reference_timestamp")

## 3. Build TAM3C2 and run the spatiotemporal analysis

In [ ]:
tam = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=max_window_ratio,
    normal_radii=normal_radii,
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=keep_neighborhoods,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

analysis = py4dgeo.SpatiotemporalAnalysis(output_path, force=True)
analysis.reference_epoch = reference_epoch
analysis.corepoints = corepoints
analysis.m3c2 = tam

analysis.add_epochs(target_epoch)
print(f"target epoch: {target_epoch.timestamp}")
print(f"epochs_timeseries used by TAM3C2: {[e.timestamp for e in tam.epochs_timeseries]}")
print(f"include_center_epoch: {tam.include_center_epoch}")
print(f"distances shape: {analysis.distances.shape}")
print(f"uncertainties shape: {analysis.uncertainties.shape}")

## 4. statistic aggregation diagnostics for the selected target

In [ ]:
diag = tam.diagnostics()
for k, v in diag.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    elif isinstance(v, list):
        print(f"  {k}: list of length {len(v)}")

# Save for later inspection
tam.save_diagnostics(os.path.join(os.getcwd(), 'simulation2_tam3c2_diag.npz'))

In [ ]:
diag['scale_idx'] # multiscale index of the scale used for each corepoint

In [ ]:
combinations = tam._scale_combinations 

i = 0  
idx = tam._opt_scale_idx[i]
print(f"corepoint {i} scale: {combinations[idx]}")

In [ ]:
diag['window_used_ref'] # unit = seconds, the time window used for aggregation, 86400=1 day, 最远 Epoch”距离中心点的时间跨度

In [ ]:
diag['window_used_ref'][0]

In [ ]:
diag['n_after_ref'] # number of epochs after reference aggregation

In [ ]:
diag['n_before_ref']

In [ ]:
diag['n_points_ref']

In [ ]:
tam._neighborhoods[0]

## visual analysis of aggregation

In [ ]:
# Visual diagnostics for temporal aggregation around each corepoint.
# Requires `tam`, `diag`, `reference_epoch`, `target_epoch`, and `corepoints` from previous cells.

def _diag_col(name, col=0):
    value = diag[name]
    arr = np.asarray(value)
    return arr[:, col] if arr.ndim == 2 else arr


def _plot_cp_map(ax, values, title, cmap='viridis', s=3, vmin=None, vmax=None, discrete=False):
    sc = ax.scatter(corepoints[:, 0], corepoints[:, 1], c=values, s=s, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel('X [m]')
    ax.set_ylabel('Y [m]')
    ax.axis('equal')
    cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    if discrete:
        vals = np.unique(np.asarray(values)[np.isfinite(values)])
        if len(vals) <= 12:
            cbar.set_ticks(vals)
    return sc


def _count_cylinder_points_in_epoch(cp, normal, epoch_idx, cyl_radius, max_distance):
    bounding_r = np.sqrt(cyl_radius * cyl_radius + max_distance * max_distance)
    idxs = tam._kdtrees[epoch_idx].query_ball_point(cp, bounding_r)
    if not idxs:
        return 0
    pts = tam.epochs_timeseries[epoch_idx].cloud[idxs]
    vec = pts - cp
    along = vec @ normal
    perp_sq = np.einsum('ij,ij->i', vec, vec) - along * along
    mask = (perp_sq <= cyl_radius * cyl_radius) & (np.abs(along) <= max_distance)
    return int(np.count_nonzero(mask))


tam._build_index()
target_col = 0
scale_idx = np.asarray(diag['scale_idx']).astype(int)
combos = tam._scale_combinations

ref_idx_in_ts = tam._find_epoch_index(reference_epoch)
tgt_idx_in_ts = tam._find_epoch_index(target_epoch)
ref_time = reference_epoch.timestamp.timestamp()
time_range = float(tam._epoch_times.max() - tam._epoch_times.min()) or 1.0
ref_exclude_idx = None if tam.include_center_epoch else ref_idx_in_ts

n_points_ref_after = _diag_col('n_points_ref', target_col)
n_points_tgt_after = _diag_col('n_points_tgt', target_col)

# Exact number of contributing epochs if keep_neighborhoods=True; fallback to diagnostics otherwise.
nbhd = tam._neighborhoods[target_col] if tam._neighborhoods is not None else None
if nbhd is not None:
    n_epochs_ref = np.array([
        len(np.unique(record['ref_eidx'])) if record is not None else 0
        for record in nbhd
    ], dtype=float)
    n_epochs_tgt = np.array([
        len(np.unique(record['tgt_eidx'])) if record is not None else 0
        for record in nbhd
    ], dtype=float)
else:
    n_epochs_ref = _diag_col('n_before_ref', target_col) + _diag_col('n_after_ref', target_col)
    n_epochs_tgt = _diag_col('n_before_tgt', target_col) + _diag_col('n_after_tgt', target_col)
    print('keep_neighborhoods=False: epoch-count maps use n_before + n_after diagnostics.')

# Center-epoch-only cylinder counts before temporal aggregation.
n_points_ref_before = np.zeros(len(corepoints), dtype=int)
n_points_tgt_before = np.zeros(len(corepoints), dtype=int)
planarity_selected = np.full(len(corepoints), np.nan)

for i, cp in enumerate(corepoints):
    normal = tam._ref_normals[i]
    sr, wr = combos[scale_idx[i]]

    n_points_ref_before[i] = _count_cylinder_points_in_epoch(
        cp, normal, ref_idx_in_ts, tam.cyl_radius, tam.max_distance
    )
    n_points_tgt_before[i] = _count_cylinder_points_in_epoch(
        cp, normal, tgt_idx_in_ts, tam.cyl_radius, tam.max_distance
    )

    pts, *_ = tam._aggregate_sphere(cp, ref_time, ref_exclude_idx, sr, time_range * wr)
    if pts is not None and len(pts) >= 3:
        planarity_selected[i], _ = tam._planarity_and_normal(pts)

# Required-points status: 0=neither side, 1=ref only, 2=target only, 3=both sides.
required_before = (
    (n_points_ref_before >= required_points).astype(int)
    + 2 * (n_points_tgt_before >= required_points).astype(int)
)
required_after = (
    (n_points_ref_after >= required_points).astype(int)
    + 2 * (n_points_tgt_after >= required_points).astype(int)
)

print('Scale index -> (normal_radius, max_window_ratio)')
for idx, combo in enumerate(combos):
    print(f'  {idx}: {combo}')
print('Required status code: 0=neither side, 1=ref only, 2=target only, 3=both sides')

# 1. Aggregation epoch-count maps.
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
vmax_epochs = max(np.nanmax(n_epochs_ref), np.nanmax(n_epochs_tgt))
_plot_cp_map(axs[0], n_epochs_ref, 'Reference aggregation: contributing epochs', 'viridis', vmax=vmax_epochs, discrete=True)
_plot_cp_map(axs[1], n_epochs_tgt, 'Target aggregation: contributing epochs', 'viridis', vmax=vmax_epochs, discrete=True)
plt.tight_layout()
plt.show()

# 2. Point-count maps before vs after temporal aggregation.
fig, axs = plt.subplots(2, 2, figsize=(13, 10))
vmax_points = np.nanpercentile(
    np.r_[n_points_ref_before, n_points_tgt_before, n_points_ref_after, n_points_tgt_after],
    98,
)
_plot_cp_map(axs[0, 0], n_points_ref_before, 'Reference center epoch: cylinder points', 'magma', vmax=vmax_points)
_plot_cp_map(axs[0, 1], n_points_ref_after, 'Reference after temporal aggregation: points used', 'magma', vmax=vmax_points)
_plot_cp_map(axs[1, 0], n_points_tgt_before, 'Target center epoch: cylinder points', 'magma', vmax=vmax_points)
_plot_cp_map(axs[1, 1], n_points_tgt_after, 'Target after temporal aggregation: points used', 'magma', vmax=vmax_points)
plt.tight_layout()
plt.show()

# 3. Required-points status before and after aggregation.
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
_plot_cp_map(axs[0], required_before, f'Before aggregation: required_points={required_points}', 'tab10', vmin=0, vmax=3, discrete=True)
_plot_cp_map(axs[1], required_after, f'After aggregation: required_points={required_points}', 'tab10', vmin=0, vmax=3, discrete=True)
plt.tight_layout()
plt.show()

# 4. Multi-scale winning index and selected-neighborhood planarity.
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
_plot_cp_map(axs[0], scale_idx, 'Winning multi-scale combination index', 'tab20', vmin=0, vmax=max(len(combos) - 1, 1), discrete=True)
_plot_cp_map(axs[1], planarity_selected, 'Planarity at selected scale', 'viridis')
plt.tight_layout()
plt.show()

## 5. Multi-scale analysis 

In [ ]:
# Full-corepoint multi-scale planarity map on the (normal_radii x max_window_ratio) grid.
#
# Important: the mean-planarity panel below does NOT run full TAM3C2 once per
# combo. It recomputes the spherical normal-estimation neighborhood for every
# (scale combo, corepoint) pair and reports the PCA planarity. This is the same
# criterion used by TAM3C2's scale-selection contest.
#
# The selection-frequency panel comes from the already-run multiscale TAM3C2
# cache: each corepoint votes for the combo selected by TAM3C2.
#
# The selected best single-scale combo is then run once as a separate TAM3C2
# analysis and saved for later comparison.

radii  = list(tam.normal_radii)     if hasattr(tam.normal_radii,     '__iter__') else [tam.normal_radii]
ratios = list(tam.max_window_ratio) if hasattr(tam.max_window_ratio, '__iter__') else [tam.max_window_ratio]
combos = tam._scale_combinations    # ordered: for sr in radii: for wr in ratios

ref_idx_in_ts = tam._find_epoch_index(reference_epoch)
ref_exclude_idx = None if tam.include_center_epoch else ref_idx_in_ts
ref_time = reference_epoch.timestamp.timestamp()
time_range = float(tam._epoch_times.max() - tam._epoch_times.min()) or 1.0

# --- 1. planarity for every (combo, corepoint), no subsampling --------------
n_corepoints = len(corepoints)
planarity = np.full((len(combos), n_corepoints), np.nan)

for k, (sr, wr) in enumerate(combos):
    mw = time_range * wr
    print(f"Computing planarity for combo {k}/{len(combos)-1}: normal_radius={sr:g}, max_window_ratio={wr:g}")
    for i, cp in enumerate(corepoints):
        pts, *_ = tam._aggregate_sphere(cp, ref_time, ref_exclude_idx, sr, mw)
        if pts is None or len(pts) < 3:
            continue
        pl, _ = tam._planarity_and_normal(pts)
        planarity[k, i] = pl

mean_pl_flat = np.nanmean(planarity, axis=1)
mean_pl = mean_pl_flat.reshape(len(radii), len(ratios))

# --- 2. selection frequency from cached multiscale diagnostics --------------
diag = tam.diagnostics()
scale_idx_all = np.asarray(diag['scale_idx']).astype(int)
counts = np.bincount(scale_idx_all, minlength=len(combos))
freq_flat = counts / counts.sum()
freq = freq_flat.reshape(len(radii), len(ratios))
combo_idx_grid = np.arange(len(combos)).reshape(len(radii), len(ratios))

# --- 3. choose one best combo and save a single-scale TAM3C2 result ----------
best_mean_planarity_idx = int(np.nanargmax(mean_pl_flat))
most_selected_idx = int(np.argmax(counts))

# Default strategy: use the scale most often selected by the per-corepoint
# multiscale contest. Change to "mean_planarity" if you want the globally
# highest mean-planarity cell instead.
best_combo_strategy = "most_selected" #！！！
if best_combo_strategy == "mean_planarity":
    best_combo_idx = best_mean_planarity_idx
elif best_combo_strategy == "most_selected":
    best_combo_idx = most_selected_idx
else:
    raise ValueError(f"Unknown best_combo_strategy: {best_combo_strategy!r}")

best_sr, best_wr = combos[best_combo_idx]
best_scale_output_path = os.path.join(
    os.getcwd(),
    f"simulation2_best_scale_idx{best_combo_idx}_tam3c2.zip",
)

print("\nScale index -> (normal_radius, max_window_ratio)")
for idx, combo in enumerate(combos):
    print(f"  {idx}: {combo}")
print(f"\nBest by mean planarity : idx={best_mean_planarity_idx}, combo={combos[best_mean_planarity_idx]}, mean_planarity={mean_pl_flat[best_mean_planarity_idx]:.4f}")
print(f"Best by selection freq : idx={most_selected_idx}, combo={combos[most_selected_idx]}, frequency={freq_flat[most_selected_idx]:.1%}")
print(f"Using best_combo_strategy={best_combo_strategy!r}: idx={best_combo_idx}, combo=({best_sr}, {best_wr})")

best_scale_tam = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=float(best_wr),
    normal_radii=float(best_sr),
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=False,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

best_scale_analysis = py4dgeo.SpatiotemporalAnalysis(best_scale_output_path, force=True)
best_scale_analysis.reference_epoch = reference_epoch
best_scale_analysis.corepoints = corepoints
best_scale_analysis.m3c2 = best_scale_tam
best_scale_analysis.add_epochs(target_epoch)
best_scale_tam.save_diagnostics(
    os.path.join(os.getcwd(), f"simulation2_best_scale_idx{best_combo_idx}_tam3c2_diag.npz")
)

multi_dist = analysis.distances[:, 0]
best_dist = best_scale_analysis.distances[:, 0]
compare_valid = np.isfinite(multi_dist) & np.isfinite(best_dist)
print(f"Saved best single-scale TAM3C2 analysis: {best_scale_output_path}")
print(f"Best single-scale distances shape: {best_scale_analysis.distances.shape}")
print(f"Compared to per-corepoint multiscale TAM3C2 on {compare_valid.sum()} valid corepoints:")
print(f"  mean(best - multiscale) = {np.nanmean(best_dist[compare_valid] - multi_dist[compare_valid]):.4f} m")
print(f"  MAE(best vs multiscale) = {np.nanmean(np.abs(best_dist[compare_valid] - multi_dist[compare_valid])):.4f} m")

# --- 4. two-panel heatmap with combo index labels ---------------------------
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for ax, M, title, cmap, fmt in [
    (axs[0], mean_pl, 'Mean planarity over all corepoints', 'viridis', '.3f'),
    (axs[1], freq,    'Selection frequency of winning scale', 'magma', '.1%'),
]:
    im = ax.imshow(M, origin='lower', aspect='auto', cmap=cmap)
    ax.set_xticks(range(len(ratios)))
    ax.set_xticklabels([f'{r:g}' for r in ratios])
    ax.set_yticks(range(len(radii)))
    ax.set_yticklabels([f'{r:g}' for r in radii])
    ax.set_xlabel('max_window_ratio')
    ax.set_ylabel('normal_radii [m]')
    ax.set_title(title)
    for row in range(M.shape[0]):
        for col in range(M.shape[1]):
            val = M[row, col]
            idx = combo_idx_grid[row, col]
            if not np.isnan(val):
                ax.text(
                    col,
                    row,
                    f"idx {idx}\n{format(val, fmt)}",
                    ha='center',
                    va='center',
                    color='white',
                    fontsize=9,
                )
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for ax in axs:
    best_row, best_col = np.argwhere(combo_idx_grid == best_combo_idx)[0]
    ax.scatter(best_col, best_row, s=220, facecolors='none', edgecolors='cyan', linewidths=2.5)

plt.tight_layout()
plt.show()

## 7. Compare against reference distances from `simulation2_reference.zip`

Load the precomputed analysis archive, select the column matching the current target epoch, align corepoints, and compare the current TAM3C2 distance estimates against the reference distances.

In [ ]:
ref_analysis.smoothed_distances

In [ ]:
indices = np.nonzero(ref_analysis.distances)[0]
np.unique(indices)

In [ ]:
# Recompute and save the no-weight best single-scale TAM3C2 analysis.
# This uses the best scale selected in the previous multiscale-analysis cell,
# but turns temporal weighting off for both reference and target aggregation.

no_weight_output_path = os.path.join(
    os.getcwd(),
    f"simulation2_best_scale_idx{best_combo_idx}_unweighted_tam3c2.zip",
)

no_weight_tam = TAM3C2(
    epochs_timeseries=epochs_all,
    max_window_ratio=float(best_wr),
    normal_radii=float(best_sr),
    required_points=required_points,
    weighting=Weighting.NONE,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    include_center_epoch=include_center_epoch,
    keep_neighborhoods=False,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

no_weight_analysis = py4dgeo.SpatiotemporalAnalysis(no_weight_output_path, force=True)
no_weight_analysis.reference_epoch = reference_epoch
no_weight_analysis.corepoints = corepoints
no_weight_analysis.m3c2 = no_weight_tam
no_weight_analysis.add_epochs(target_epoch)

no_weight_diag_path = os.path.join(
    os.getcwd(),
    f"simulation2_best_scale_idx{best_combo_idx}_unweighted_tam3c2_diag.npz",
)
no_weight_tam.save_diagnostics(no_weight_diag_path)

print(f"Saved unweighted best-scale TAM3C2 analysis: {no_weight_output_path}")
print(f"Saved unweighted diagnostics: {no_weight_diag_path}")
print(f"Unweighted distances shape: {no_weight_analysis.distances.shape}")
print(f"Unweighted uncertainties shape: {no_weight_analysis.uncertainties.shape}")

In [ ]:
# Compute and save a direct M3C2 analysis on the selected reference/target pair.
# This is the standard two-epoch M3C2 baseline without temporal aggregation.
# It uses the same corepoints, cylinder radius, max distance, and best normal
# radius selected from the TAM3C2 multiscale analysis.

direct_m3c2_output_path = os.path.join(
    os.getcwd(),
    f"simulation2_best_scale_idx{best_combo_idx}_direct_m3c2.zip",
)

direct_m3c2 = py4dgeo.M3C2(
    epochs=(reference_epoch, target_epoch),
    corepoints=corepoints,
    normal_radii=[float(best_sr)],
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

direct_m3c2_analysis = py4dgeo.SpatiotemporalAnalysis(direct_m3c2_output_path, force=True)
direct_m3c2_analysis.reference_epoch = reference_epoch
direct_m3c2_analysis.corepoints = corepoints
direct_m3c2_analysis.m3c2 = direct_m3c2
direct_m3c2_analysis.add_epochs(target_epoch)

print(f"Saved direct M3C2 analysis: {direct_m3c2_output_path}")
print(f"Direct M3C2 distances shape: {direct_m3c2_analysis.distances.shape}")
print(f"Direct M3C2 uncertainties shape: {direct_m3c2_analysis.uncertainties.shape}")

In [ ]:
# Plot four error maps and four roughness maps.
#
# Error is defined as: method distance - mesh-reference distance.
# Roughness is represented by the mean M3C2 spread per corepoint:
# 0.5 * (spread1 + spread2), where spread1 is the reference-side spread and
# spread2 is the target-side spread.

from scipy.spatial import cKDTree


def _target_column_from_reference(reference_analysis, target_time, reference_time):
    expected_delta = target_time - reference_time
    timedeltas = list(reference_analysis.timedeltas)
    matches = [idx for idx, delta in enumerate(timedeltas) if delta == expected_delta]
    if matches:
        return matches[0]

    delta_seconds = np.array([abs((delta - expected_delta).total_seconds()) for delta in timedeltas])
    nearest = int(np.argmin(delta_seconds))
    print(
        f"No exact reference timedelta match for {expected_delta}; "
        f"using nearest column {nearest} ({timedeltas[nearest]})."
    )
    return nearest


def _aligned_reference_distance(reference_analysis, query_corepoints, target_time, reference_time):
    target_col = _target_column_from_reference(reference_analysis, target_time, reference_time)
    reference_corepoints = reference_analysis.corepoints.cloud
    reference_distance_all = reference_analysis.distances[:, target_col]

    if len(reference_corepoints) == len(query_corepoints) and np.allclose(reference_corepoints, query_corepoints):
        return reference_distance_all, np.arange(len(query_corepoints)), target_col

    tree = cKDTree(reference_corepoints[:, :3])
    distances_to_ref, reference_idx = tree.query(query_corepoints[:, :3], k=1)
    print(
        "Corepoints are not identical; using nearest reference corepoint alignment. "
        f"median distance={np.median(distances_to_ref):.4f} m, "
        f"max={np.max(distances_to_ref):.4f} m"
    )
    return reference_distance_all[reference_idx], reference_idx, target_col


def _distance_column(st_analysis, label):
    if st_analysis.distances is None or st_analysis.distances.shape[1] == 0:
        raise ValueError(f"{label} has no distance column")
    return st_analysis.distances[:, 0].astype(float)


def _roughness_from_uncertainty(st_analysis, label):
    if st_analysis.uncertainties is None:
        raise ValueError(f"{label} has no uncertainty array")

    uncertainty = st_analysis.uncertainties[:, 0]
    names = uncertainty.dtype.names or ()
    if "spread1" not in names or "spread2" not in names:
        raise ValueError(f"{label} uncertainty does not contain spread1/spread2 fields: {names}")

    return 0.5 * (uncertainty["spread1"].astype(float) + uncertainty["spread2"].astype(float))


def _finite_percentile(values, percentile, default=1.0):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return default
    limit = float(np.nanpercentile(finite, percentile))
    if not np.isfinite(limit) or limit == 0:
        return default
    return limit


def _format_map_axes(ax):
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_aspect("equal", adjustable="box")


reference_distance, reference_indices, reference_target_col = _aligned_reference_distance(
    ref_analysis,
    corepoints,
    target_timestamp,
    reference_timestamp,
)
print(f"Using reference distance column {reference_target_col} for target {target_timestamp}")

comparison_results = {
    "Best scale weighted": {
        "distance": _distance_column(best_scale_analysis, "Best scale weighted"),
        "roughness": _roughness_from_uncertainty(best_scale_analysis, "Best scale weighted"),
    },
    "Multiscale weighted": {
        "distance": _distance_column(analysis, "Multiscale weighted"),
        "roughness": _roughness_from_uncertainty(analysis, "Multiscale weighted"),
    },
    "Best scale unweighted": {
        "distance": _distance_column(no_weight_analysis, "Best scale unweighted"),
        "roughness": _roughness_from_uncertainty(no_weight_analysis, "Best scale unweighted"),
    },
    "Direct M3C2": {
        "distance": _distance_column(direct_m3c2_analysis, "Direct M3C2"),
        "roughness": _roughness_from_uncertainty(direct_m3c2_analysis, "Direct M3C2"),
    },
}

for label, result in comparison_results.items():
    result["error"] = result["distance"] - reference_distance
    valid_error = np.isfinite(result["error"])
    valid_roughness = np.isfinite(result["roughness"])
    print(
        f"{label}: valid error={valid_error.sum():,}/{len(valid_error):,}, "
        f"mean error={np.nanmean(result['error']):.4f} m, "
        f"MAE={np.nanmean(np.abs(result['error'])):.4f} m, "
        f"mean roughness={np.nanmean(result['roughness']):.4f} m "
        f"({valid_roughness.sum():,} valid)"
    )

xy = corepoints[:, :2]
error_values = np.concatenate([result["error"] for result in comparison_results.values()])
roughness_values = np.concatenate([result["roughness"] for result in comparison_results.values()])
error_limit = _finite_percentile(np.abs(error_values), 98, default=0.1)
roughness_limit = _finite_percentile(roughness_values, 98, default=0.1)

fig, axs = plt.subplots(1, 4, figsize=(22, 5), constrained_layout=True)
for ax, (label, result) in zip(axs, comparison_results.items()):
    scatter = ax.scatter(
        xy[:, 0],
        xy[:, 1],
        c=result["error"],
        s=3,
        cmap="seismic_r",
        vmin=-error_limit,
        vmax=error_limit,
    )
    ax.set_title(f"{label}\nerror = method - reference")
    _format_map_axes(ax)
    fig.colorbar(scatter, ax=ax, label="Error [m]", fraction=0.046, pad=0.04)
plt.show()

fig, axs = plt.subplots(1, 4, figsize=(22, 5), constrained_layout=True)
for ax, (label, result) in zip(axs, comparison_results.items()):
    scatter = ax.scatter(
        xy[:, 0],
        xy[:, 1],
        c=result["roughness"],
        s=3,
        cmap="viridis",
        vmin=0,
        vmax=roughness_limit,
    )
    ax.set_title(f"{label}\nroughness = mean(spread1, spread2)")
    _format_map_axes(ax)
    fig.colorbar(scatter, ax=ax, label="Spread roughness [m]", fraction=0.046, pad=0.04)
plt.show()